# 12 · DeBERTa-v3-large on 2 × T4 (DDP): full run and 5-fold OOF

**Why:** DeBERTa-v3-large is the one backbone likely to beat DeBERTa-v3-base (`d2_deberta_main_ce4_s42`, val 0.8891). On one T4 a full epoch of `main`/256 is estimated at ~7.5–8 h plus ~70 min of prediction, and batch 32 does not fit in 16 GB. Kaggle's **GPU T4 × 2** splits the work over two GPUs.

**How:** the notebook writes a training script (section 2) and starts it with `torchrun`, one process per GPU (PyTorch DistributedDataParallel). It runs the same model, ce4 loss, optimizer, schedule and fp16 as `src/cross_encoder.train_ce`; only the batch is split:
- Global batch 32 = 2 GPUs × 16 pairs. Each GPU's 16 pairs are cut into micro-batches of `micro_batch` (4 for large: 4 micro-batches with gradient accumulation; 8 ran out of memory) so that they fit in memory.
- Gradients are averaged over both GPUs before clipping, so one step is the same update as one single-GPU step on the whole batch of 32 (same number of steps, same schedule).
- Prediction: the rows are split between the GPUs, and the notebook puts the scores back in row order.

**Modes** (`MODE` in the config cell):

| `MODE` | Trains on | Predicts | Use |
|---|---|---|---|
| `smoke` | 5% of train queries | val | check code, memory, speed (base: vs the 1-GPU smoke 0.8705) |
| `full` | all train (378k pairs, 11,816 steps) | val, test (blind) | `d1_debertalarge_main_ce4_s42`: the large model itself (done: val 0.8946) |
| `oof` | 4 of the 5 folds (~302k pairs, ~9,450 steps), for each fold in `FOLDS` | held-out fold (`oof`), val, test (blind) | `oof_debertalarge_main_ce4_f<k>`: input of the LightGBM re-ranker, as `07_oof` for DeBERTa-base |
| `assemble` | – (CPU) | – | joins the 5 folds into `oof/ce_debertalarge_main_ce4.parquet` |

`oof` uses the folds of `07_oof` (`fold = fold10 // 2`, by `dup_group`): every train pair gets a score from the model that did not see its query, and val/test get the mean of the 5 fold models, so the re-ranker sees train, val and test scores of the same kind. No val curve during fold training (saves ~10 min per fold); each fold model is compared on val with the full large run.

**Large settings:** as `d2_deberta_main_ce4_s42` (`main`, `max_len` 256, ce4, 1 epoch, batch 32, 1,200 warm-up steps, fp16, seed 42) except lr **1e-5** instead of 2e-5 (large models are usually fine-tuned with a smaller lr; not tuned here).

**Time (large, 2 × T4, measured on the full run: 0.69 it/s, 30 min to predict val + test):** full ~5.3 h; one fold ~4.5 h (3.8 h training + ~40 min predicting fold, val and test). Two folds per version (~9 h) fit the 12 h session limit: e.g. account 1 `FOLDS = [0, 1]`, account 2 `FOLDS = [2, 3]`, then `[4]`. Run `full` and `oof` with *Save Version → Save & Run All*; each fold's results are written as soon as it ends, so a stopped session keeps the folds already done.

**Kaggle settings:** Accelerator **GPU T4 × 2**, **Internet on** (Hugging Face model); `assemble` runs on CPU without internet.

**Inputs:**
- Dataset `wm-src` (`src/` and `data/results/`). Required; `data/results/` holds `d2_deberta_main_ce4_s42` and, for `oof`, `d1_debertalarge_main_ce4_s42` for the paired comparisons on val.
- Notebook Output → **01_preprocess** (`pairs.parquet`, `products_clean.parquet`). Required.
- `assemble` only: the outputs of the versions that ran the 5 folds.

**Output** (`/kaggle/working/results/`): `<run>.json` (config, loss log, val curve, speed, memory), `<run>_train_log.txt` and `scores/<run>_<split>.parquet` with `p_E..p_I`; after `assemble`: `oof/ce_debertalarge_main_ce4.parquet`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import json, os, shutil, subprocess, time

import numpy as np
import pandas as pd
import torch
import transformers
from huggingface_hub import snapshot_download

from src.cross_encoder import LABELS
from src.data import load_pairs
from src.evaluation import RunLog, summarize

MODE = 'oof'     # 'smoke' | 'full' | 'oof' (fold models of FOLDS) | 'assemble' (CPU: join the 5 folds); see the table above
SIZE = 'large'   # 'base': check the 2-GPU path against the 1-GPU runs | 'large': DeBERTa-v3-large
FOLDS = [0, 1]   # oof mode: folds trained in this version (0-4); ~4.5 h each for large

SIZES = {'base': {'model': 'microsoft/deberta-v3-base', 'lr': 2e-5, 'micro_batch': 16, 'eval_batch_size': 64,
                  'name': 'd1_deberta_main_ce4_ddp', 'smoke_ref': 0.8705},
         'large': {'model': 'microsoft/deberta-v3-large', 'lr': 1e-5, 'micro_batch': 4, 'eval_batch_size': 32,
                   'name': 'd1_debertalarge_main_ce4', 'smoke_ref': None}}   # large: 8 ran out of memory on a T4
Z = SIZES[SIZE]
CFG = {'epochs': 1, 'batch_size': 32, 'weight_decay': 0.01, 'max_grad_norm': 1.0, 'fp16': True,
       'model': Z['model'], 'fields': 'main', 'max_len': 256, 'lr': Z['lr'], 'warmup_steps': 1200,
       'eval_batch_size': Z['eval_batch_size'], 'loss': 'ce4',
       'micro_batch': Z['micro_batch'],   # pairs per forward pass on one GPU (gradient accumulation up to batch_size / GPUs)
       'grad_ckpt': False,                # gradient checkpointing: set True only if a GPU runs out of memory (slower)
       'find_unused': False}              # DDP option: DeBERTa uses every parameter (checked: DDP found none unused)

SEED = 42
N_FOLDS = 5                                                      # fold = fold10 // 2, as in 07_oof
NAME = f"{'smoke_' if MODE == 'smoke' else ''}{Z['name']}_s{SEED}"
OOF_NAME = 'oof_' + Z['name'][3:]                                # oof_debertalarge_main_ce4 -> runs <OOF_NAME>_f<k>
# paired comparison on val: smoke/full with DeBERTa-base (same settings), fold models with the full run of this size
REF = f"{Z['name']}_s{SEED}" if MODE == 'oof' else 'd2_deberta_main_ce4_s42'
SUBSAMPLE = 0.05 if MODE == 'smoke' else 1.0
EVAL_SPLITS = ['val'] if MODE == 'smoke' else ['val', 'test']   # test is predicted and saved, never scored here
EVAL_EVERY = {'smoke': 300, 'full': 4000}.get(MODE, 0)           # steps between val checks (monitoring only; none in oof)
PROBS = [f'p_{l}' for l in LABELS]
RES = Path('/kaggle/working/results')
WORK = Path('/tmp/ddp')                                          # script, inputs and per-GPU outputs (not saved)

assert MODE in ('smoke', 'full', 'oof', 'assemble') and SIZE in SIZES
assert set(FOLDS) <= set(range(N_FOLDS))
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
transformers.logging.set_verbosity_error()
if MODE != 'assemble':
    N_GPU = torch.cuda.device_count()   # counts the GPUs without starting CUDA in this process (the GPUs stay free)
    assert N_GPU >= 2, f'{N_GPU} GPU(s): Settings -> Accelerator -> GPU T4 x2'
    assert CFG['batch_size'] % (N_GPU * CFG['micro_batch']) == 0, 'batch_size must be a multiple of GPUs x micro_batch'
    print('torch', torch.__version__, '| transformers', transformers.__version__, '|', N_GPU, 'GPUs')
    print(f"batch {CFG['batch_size']} = {N_GPU} GPUs x {CFG['batch_size'] // N_GPU} pairs "
          f"({CFG['batch_size'] // N_GPU // CFG['micro_batch']} micro-batches of {CFG['micro_batch']})")
print('MODE', MODE, '| SIZE', SIZE, '|', CFG['model'], '| lr', CFG['lr'],
      '| runs', [f'{OOF_NAME}_f{k}' for k in FOLDS] if MODE == 'oof' else OOF_NAME if MODE == 'assemble' else NAME)

## 1. Data and jobs

Same pairs, `main` text, folds and 5% smoke sample (same queries) as the 1-GPU runs. Train pairs get `fold = fold10 // 2` (a query sits in one fold). One job = one training run: its training pairs and the splits it predicts (`oof` mode: one job per fold in `FOLDS`). The model weights are downloaded once here, so the two GPU processes load them from the cache.

In [ ]:
pairs = load_pairs(CFG['fields'])
pairs['fold'] = np.where(pairs['split'] == 'train', pairs['fold10'] // 2, -1)
train_all = pairs[pairs['split'] == 'train']
fixed = {s: pairs[pairs['split'] == s].reset_index(drop=True) for s in ['val', 'test']}
assert train_all.groupby('query_id')['fold'].nunique().max() == 1, 'a query is in two folds'

JOBS = []   # (run name, training pairs, {split: pairs to predict}, extra info for the run json)
if MODE in ('smoke', 'full'):
    train_df = train_all
    if SUBSAMPLE < 1:
        keep = train_df['query_id'].drop_duplicates().sample(frac=SUBSAMPLE, random_state=SEED)
        train_df = train_df[train_df['query_id'].isin(keep)]
    JOBS.append((NAME, train_df.reset_index(drop=True), {s: fixed[s] for s in EVAL_SPLITS}, {}))
elif MODE == 'oof':
    for k in FOLDS:
        JOBS.append((f'{OOF_NAME}_f{k}', train_all[train_all['fold'] != k].reset_index(drop=True),
                     {'oof': train_all[train_all['fold'] == k].reset_index(drop=True), **fixed},
                     {'fold': k, 'n_folds': N_FOLDS, 'train_folds': [f for f in range(N_FOLDS) if f != k]}))
for name, tr, parts, _ in JOBS:
    print(f"{name}: train {tr['query_id'].nunique():,} queries / {len(tr):,} pairs ({len(tr) // CFG['batch_size']:,} steps) | "
          + ' | '.join(f'{s} {len(p):,}' for s, p in parts.items()))
if JOBS:
    snapshot_download(CFG['model'], allow_patterns=['*.json', '*.bin', '*.safetensors', '*.model'])
    print('weights in the cache:', CFG['model'])

## 2. Training script (one process per GPU)

Written to `/tmp/ddp/ddp_ce.py` and started in section 3. It imports the loss, optimizer, encoding and prediction from `src/cross_encoder.py`, so only the split over GPUs is new:
- Both processes draw the same permutation of the training pairs; at each step process `r` takes its 16 pairs of the batch of 32.
- The micro-batches before the last one run under `no_sync()` (no gradient exchange); after the last one DDP averages the gradients of both GPUs. The loss of each micro-batch is divided by the number of micro-batches, so the gradient equals the mean over the 32 pairs.
- The new classification head is initialised with the same seed on both GPUs (DDP also copies GPU 0's weights to GPU 1 at the start); dropout uses a different seed per GPU.
- Val checks and the final prediction: rows sorted by length and dealt to the GPUs, each writes `<tag>_<rank>.npz`, process 0 puts them back in row order.

In [ ]:
%%writefile /tmp/ddp/ddp_ce.py
import contextlib, json, os, sys, time
from datetime import timedelta
from pathlib import Path

JOB = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
sys.path.insert(0, JOB['root'])

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import transformers
from torch.nn.parallel import DistributedDataParallel as DDP
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from src.cross_encoder import LABELS, compute_loss, encode, make_optimizer, make_scaler, predict, set_seed
from src.evaluation import summarize

transformers.logging.set_verbosity_error()
dist.init_process_group('nccl', timeout=timedelta(hours=2))
rank, world = dist.get_rank(), dist.get_world_size()
local = int(os.environ['LOCAL_RANK'])
torch.cuda.set_device(local)   # 'cuda' in src.cross_encoder now means this process's GPU
main = rank == 0
cfg, seed, out = JOB['cfg'], JOB['seed'], Path(JOB['out_dir'])
ce4 = cfg['loss'] == 'ce4'
assert cfg['loss'] in ('ce4', 'mse'), 'this script supports ce4 and mse (ListNet needs query batches)'


def say(msg):
    if main:
        print(msg, flush=True)
        with open(out / 'train_log.txt', 'a', encoding='utf-8') as f:
            f.write(msg + '\n')


train = pd.read_parquet(JOB['train'])
evals = {s: pd.read_parquet(p) for s, p in JOB['eval'].items()}
q, t = train['query'].to_numpy(), train['text'].to_numpy()
if ce4:
    y = train['esci_label'].map({l: i for i, l in enumerate(LABELS)}).to_numpy(np.int64)
else:
    y = train['gain'].to_numpy(np.float32)

gb, mb = cfg['batch_size'], cfg['micro_batch']
per = gb // world        # pairs per GPU and step
accum = per // mb        # micro-batches per GPU and step
assert per * world == gb and accum * mb == per

set_seed(seed)           # same seed on both GPUs: same new head
tok = AutoTokenizer.from_pretrained(cfg['model'])
model = AutoModelForSequenceClassification.from_pretrained(cfg['model'], num_labels=len(LABELS) if ce4 else 1,
                                                           ignore_mismatched_sizes=ce4)
model = model.float().cuda()
if cfg.get('grad_ckpt'):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
# gradient_as_bucket_view: the gradients live inside DDP's communication buffers instead of a second copy
# (saves one copy of the gradients, 1.7 GB for large)
model = DDP(model, device_ids=[local], find_unused_parameters=cfg.get('find_unused', False),
            gradient_as_bucket_view=True)
set_seed(seed + 1000 * rank)   # different dropout masks on each GPU

steps_per_epoch = len(train) // gb
total = steps_per_epoch * cfg['epochs']
warmup = min(cfg['warmup_steps'], int(0.42 * total))   # as in cross_encoder.train_ce
opt = make_optimizer(model.module, cfg)
sched = get_linear_schedule_with_warmup(opt, warmup, total)
scaler = make_scaler(cfg['fp16'])
rng = np.random.default_rng(seed)   # same permutation on both GPUs; each takes its own part of a batch
perms = [rng.permutation(len(train)) for _ in range(cfg['epochs'])]
say(f'{world} GPUs x {per} pairs ({accum} x {mb}) = batch {gb} | {total} steps | warmup {warmup} | '
    f'{len(train):,} pairs | {torch.cuda.get_device_name(local)}')


def predict_part(split, tag):
    # This GPU's share of a split (rows sorted by length, dealt round-robin), written to <tag>_<rank>.npz.
    df = evals[split]
    order = np.argsort((df['query'].str.len() + df['text'].str.len()).to_numpy(), kind='stable')
    mine = order[rank::world]
    torch.cuda.empty_cache()
    r = predict(model.module, tok, df.iloc[mine], cfg['max_len'], batch_size=cfg.get('eval_batch_size', 32),
                fp16=cfg['fp16'], return_probs=ce4)
    save = {'idx': mine, 'score': r[0] if ce4 else r}
    if ce4:
        save['probs'] = r[1].to_numpy()
    np.savez(out / f'{tag}_{rank}.npz', **save)
    dist.barrier()


def val_score(tag):
    score = np.full(len(evals['val']), np.nan, dtype=np.float32)
    for r in range(world):
        z = np.load(out / f'{tag}_{r}.npz')
        score[z['idx']] = z['score']
    assert not np.isnan(score).any(), f'{tag}: rows missing'
    m, _ = summarize(evals['val'], score)
    return m


def peak_gb():
    m = torch.tensor(torch.cuda.max_memory_allocated() / 2 ** 30, device='cuda')
    dist.all_reduce(m, op=dist.ReduceOp.MAX)
    return round(m.item(), 1)


loss_log, val_curve = [], []
run_loss = torch.zeros((), device='cuda')
step, eval_sec, t0, first_step_sec = 0, 0.0, time.time(), None
model.train()
for ep in range(cfg['epochs']):
    perm = perms[ep]
    for b in range(steps_per_epoch):
        idx = perm[b * gb + rank * per: b * gb + (rank + 1) * per]
        opt.zero_grad(set_to_none=True)
        for k in range(accum):
            m = idx[k * mb:(k + 1) * mb]
            with model.no_sync() if k < accum - 1 else contextlib.nullcontext():
                with torch.autocast('cuda', dtype=torch.float16, enabled=cfg['fp16']):
                    logits = model(**encode(tok, q[m], t[m], cfg['max_len'])).logits
                loss = compute_loss(logits, torch.from_numpy(y[m]).cuda(), cfg) / accum
                scaler.scale(loss).backward()
            run_loss += loss.detach()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['max_grad_norm'])
        scaler.step(opt)
        scaler.update()
        sched.step()
        step += 1
        if step == 1:
            first_step_sec = round(time.time() - t0, 1)
        if step % 100 == 0 or step == total:
            n = 100 if step % 100 == 0 else step % 100
            tot = run_loss.clone()
            dist.all_reduce(tot)
            avg = tot.item() / world / n
            run_loss.zero_()
            assert np.isfinite(avg), f'loss is not finite at step {step}'
            el = time.time() - t0 - eval_sec   # training time only, val checks excluded
            loss_log.append({'step': step, 'loss': round(avg, 6), 'lr': sched.get_last_lr()[0]})
            if step == 100 or step % JOB['print_every'] == 0 or step == total:
                say(f'step {step:>6}/{total} | loss {avg:.5f} | {step / el:.2f} it/s | {el / 60:.1f} min, '
                    f'~{(total - step) / (step / el) / 60:.0f} min left | peak memory {peak_gb()} GB/GPU')
        if 'val' in evals and JOB['eval_every'] and step % JOB['eval_every'] == 0 and step < total:
            te = time.time()
            predict_part('val', f'val_{step}')
            if main:
                mv = val_score(f'val_{step}')
                val_curve.append({'step': step, 'val_ndcg': mv['ndcg'], 'val_e_over_s': mv['e_over_s']})
                say(f'   val @ {step}: nDCG {mv["ndcg"]} | E-over-S {mv["e_over_s"]}')
                for f in out.glob(f'val_{step}_*.npz'):
                    f.unlink()
            eval_sec += time.time() - te
train_sec = time.time() - t0 - eval_sec
peak = peak_gb()

tp = time.time()
for split in evals:
    predict_part(split, f'final_{split}')
if main:
    if 'val' in evals:   # the last point of the val curve comes from the final prediction
        mv = val_score('final_val')
        val_curve.append({'step': step, 'val_ndcg': mv['ndcg'], 'val_e_over_s': mv['e_over_s']})
        say(f'   val @ {step}: nDCG {mv["ndcg"]} | E-over-S {mv["e_over_s"]}')
    info = {'steps': step, 'warmup': warmup, 'world_size': world, 'per_gpu_batch': per, 'micro_batch': mb,
            'accum': accum, 'train_minutes': round(train_sec / 60, 1), 'eval_minutes': round(eval_sec / 60, 1),
            'predict_minutes': round((time.time() - tp) / 60, 1), 'it_per_s': round(step / train_sec, 2),
            'first_step_seconds': first_step_sec, 'peak_memory_gb': peak,
            'gpu': f'{world} x {torch.cuda.get_device_name(local)}', 'loss_log': loss_log, 'val_curve': val_curve}
    (out / 'info.json').write_text(json.dumps(info), encoding='utf-8')
    say(f'done | train {info["train_minutes"]} min | predict {info["predict_minutes"]} min')
dist.barrier()
dist.destroy_process_group()

## 3. Run, gather and log

For each job: write its pairs to `/tmp/ddp/<run>/`, start both GPU processes with `torchrun` (process 0's progress is printed here), put the per-GPU outputs back in row order, and log the run with `RunLog` like every other run (test saved blind, `p_E..p_I` next to the score). A job's results are written before the next job starts. `peak memory` is the larger of the two GPUs (a T4 has 15 GB usable). If a run stops with *CUDA out of memory*, set `grad_ckpt` to True (or `micro_batch` to 2) and run again; if the loss is not finite, send me the output.

In [ ]:
COLS = ['example_id', 'query_id', 'query', 'text', 'esci_label', 'gain']
ENV = {**os.environ, 'TOKENIZERS_PARALLELISM': 'false', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
       'TRANSFORMERS_VERBOSITY': 'error', 'OMP_NUM_THREADS': '2'}


def gather(out, tag, n):
    score, probs = np.full(n, np.nan, dtype=np.float32), np.full((n, len(LABELS)), np.nan, dtype=np.float32)
    for r in range(N_GPU):
        z = np.load(out / f'{tag}_{r}.npz')
        score[z['idx']], probs[z['idx']] = z['score'], z['probs']
    assert not np.isnan(score).any() and not np.isnan(probs).any(), f'{tag}: rows missing from the GPU outputs'
    return score, pd.DataFrame(probs, columns=PROBS)


def run_job(name, tr, parts, extra):
    work = WORK / name
    (work / 'out').mkdir(parents=True)
    tr[COLS].to_parquet(work / 'train.parquet')
    for s, p in parts.items():
        p[COLS].to_parquet(work / f'{s}.parquet')
    job = {'root': str(ROOT), 'cfg': CFG, 'seed': SEED, 'out_dir': str(work / 'out'), 'eval_every': EVAL_EVERY,
           'print_every': 100 if MODE == 'smoke' else 1000, 'train': str(work / 'train.parquet'),
           'eval': {s: str(work / f'{s}.parquet') for s in parts}}
    (work / 'job.json').write_text(json.dumps(job), encoding='utf-8')

    print(f'\n==== {name}', flush=True)
    t0 = time.time()
    cmd = [sys.executable, '-m', 'torch.distributed.run', '--standalone', f'--nproc_per_node={N_GPU}',
           str(WORK / 'ddp_ce.py'), str(work / 'job.json')]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=ENV) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
    print(f'exit code {proc.returncode} | {(time.time() - t0) / 60:.1f} min')
    assert proc.returncode == 0, 'the training processes failed: send me the lines above'

    info = json.loads((work / 'out' / 'info.json').read_text(encoding='utf-8'))
    scores, probs = {}, {}
    for s, p in parts.items():
        scores[s], probs[s] = gather(work / 'out', f'final_{s}', len(p))
    log = RunLog(RES, {s: p[['example_id', 'query_id', 'esci_label']] for s, p in parts.items()})
    have_ref = MODE != 'smoke' and log.load(REF, splits=['val'])
    if MODE != 'smoke' and not have_ref:
        print(f'{REF}: not found in the inputs (no paired comparison)')
    log.log(name, scores, {'mode': MODE, **CFG, 'seed': SEED, 'subsample': SUBSAMPLE, **extra,
                           'train_queries': int(tr['query_id'].nunique()), 'train_pairs': len(tr), **info},
            compare_to=REF if have_ref else None, blind=('test',), extra=probs)
    shutil.copy(work / 'out' / 'train_log.txt', RES / f'{name}_train_log.txt')
    shutil.rmtree(work)   # free /tmp before the next job
    return log


logs = {name: run_job(name, tr, parts, extra) for name, tr, parts, extra in JOBS}

In [ ]:
if MODE == 'smoke':
    r = logs[NAME].records[NAME]
    first, last = r['loss_log'][0]['loss'], r['loss_log'][-1]['loss']
    v = r['metrics']['val']['ndcg']
    train_h = len(train_all) // CFG['batch_size'] * CFG['epochs'] / r['it_per_s'] / 3600
    pred_h = r['predict_minutes'] * (len(fixed['val']) + len(fixed['test'])) / len(fixed['val']) / 60
    ok = last < first and v > 0.742 and np.isfinite(last)
    print(f"val nDCG {v} (reference smoke {Z['smoke_ref']}) | loss {first:.4f} -> {last:.4f} | "
          f"{r['it_per_s']} it/s | peak memory {r['peak_memory_gb']} GB/GPU")
    print(f'full run estimate: train ~{train_h:.1f} h + predict val/test ~{pred_h:.1f} h '
          f'+ val checks = ~{train_h + pred_h + 0.3:.1f} h (Kaggle session limit 12 h)')
    if SIZE == 'base':
        print('2-GPU path OK: run SIZE = large, MODE = smoke.' if ok and abs(v - Z['smoke_ref']) < 0.01 else
              'Val is far from the 1-GPU smoke or the loss did not go down: send me this output.')
    else:
        print('Smoke test passed: switch MODE to full.' if ok else 'Run did not learn: send me this output.')

## 4. Results (val)

`paired_vs`: val nDCG difference with the reference over the same queries (`full`: DeBERTa-base, same settings; `oof`: the full large run, i.e. what training on 80% of train costs). Fold models also show their nDCG on the held-out fold (`oof`, train labels). Test scores are saved blind and scored in `10_report`.

In [ ]:
for name, log in logs.items():
    if MODE != 'smoke':
        display(log.table(('ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s')))
        log.print_paired()
    curve = pd.DataFrame(log.records[name]['val_curve'])
    if len(curve):
        display(curve.set_index('step'))

## 5. Assemble the 5 folds (`MODE = 'assemble'`)

Attach the outputs of every version that ran folds (or a dataset with their `results/`). As in `07_oof`, for each pair:
- **train**: score and probabilities of the one model that held its fold out;
- **val, test**: mean over the 5 fold models (probabilities averaged, score = expected gain of the averaged probabilities).

Checks: every train pair covered exactly once, by the model of its own fold; val/test covered by all 5 models. The table compares the train OOF nDCG with the val nDCG of the averaged models and of each fold model.

In [ ]:
if MODE == 'assemble':
    def find_scores(k, split):
        hits = sorted(glob.glob(f'/kaggle/input/**/scores/{OOF_NAME}_f{k}_{split}.parquet', recursive=True)
                      + glob.glob(str(RES / 'scores' / f'{OOF_NAME}_f{k}_{split}.parquet')))
        assert hits, f'{OOF_NAME}_f{k}_{split}.parquet not found: attach the output of the version that ran fold {k}'
        return pd.read_parquet(hits[0])

    gain = {'E': 1.0, 'S': 0.1, 'C': 0.01, 'I': 0.0}
    oof = pd.concat([find_scores(k, 'oof').assign(fold=k) for k in range(N_FOLDS)], ignore_index=True)
    t = train_all[['example_id', 'fold']].merge(oof[['example_id', 'fold']], on='example_id', how='left', suffixes=('', '_model'))
    assert oof['example_id'].is_unique and len(oof) == len(train_all), 'train pairs missing or scored twice'
    assert (t['fold'] == t['fold_model']).all(), 'a train pair was scored by a model that saw its query'

    out, rows = [oof.drop(columns='fold').assign(split='train')], []
    for s in ['val', 'test']:
        per = [find_scores(k, s).set_index('example_id').loc[fixed[s]['example_id']] for k in range(N_FOLDS)]
        mean = sum(p[PROBS] for p in per) / N_FOLDS
        avg = mean.assign(query_id=per[0]['query_id'], score=mean @ np.array([gain[c[-1]] for c in PROBS]))
        out.append(avg.reset_index()[['example_id', 'query_id', 'score'] + PROBS].assign(split=s))
        if s == 'val':   # test stays blind
            for k, p in enumerate(per):
                rows.append({'scores': f'fold {k} model', 'split': 'val', **summarize(fixed['val'], p['score'].to_numpy())[0]})
            rows.append({'scores': 'mean of 5 models', 'split': 'val', **summarize(fixed['val'], avg['score'].to_numpy())[0]})
    tr_sorted = train_all[['example_id', 'query_id', 'esci_label']].merge(oof[['example_id', 'score']], on='example_id')
    rows.append({'scores': 'out-of-fold', 'split': 'train', **summarize(tr_sorted, tr_sorted['score'].to_numpy())[0]})
    display(pd.DataFrame(rows)[['scores', 'split', 'ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s']])

    table = pd.concat(out, ignore_index=True)
    table[['score'] + PROBS] = table[['score'] + PROBS].astype('float32')
    assert table['example_id'].is_unique and len(table) == len(pairs) and table[['score'] + PROBS].notna().all().all()
    (RES / 'oof').mkdir(parents=True, exist_ok=True)
    path = RES / 'oof' / f"ce_{OOF_NAME[4:]}.parquet"
    table.to_parquet(path, index=False)
    print('wrote', path, table.shape, table['split'].value_counts().to_dict())